# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I'm re-ranking using my logistic regression model's **honest** (held-out, client-grouped-split) test-set probabilities.

**Important scope note:** this queue only covers the ~30% of eligible pages that landed in the honest test split (`GroupShuffleSplit`, `test_size=0.3`, `random_state=42`) — not the full dataset the Week 4 baseline ranked. Scoring the training rows would defeat the point of holding them out.

I kept the Week 4 reason codes to serve as extra information for a human reviewer, but they are **not** used as model inputs or used to train the model itself — the ranking itself comes only from `model_probability`. The reason codes just help to explain, in human readable terms, *why* a page looks risky.

Reason codes (a page can carry more than one):
- `high_visibility_at_risk`: at least 1,000 prior impressions and a weak prior position.
- `weak_position_signal`: prior average position is worse than 10.
- `low_prior_engagement`: prior engagement rate is below 30% when sessions are available.
- `low_click_through_rate`: prior clicks are low relative to prior impressions.
- `limited_prior_visibility`: fewer than 1,000 prior impressions, but still at least 100 and eligible.

This is a decision-support ranking, not a causal claim about what will happen to any given page.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load the same February-March dataset used in Weeks 4-6
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()

# Same feature prep as Week 5 / Week 6
X = dataframe[['prior_impressions', 'prior_clicks', 'prior_avg_position',
               'prior_sessions', 'prior_engagement_rate']].copy()
X['prior_avg_position'] = X['prior_avg_position'].fillna(999)
X['prior_sessions'] = X['prior_sessions'].fillna(0)
X['prior_engagement_rate'] = X['prior_engagement_rate'].fillna(0)
X['prior_ctr'] = (dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)).fillna(0)

model_features = ['prior_impressions', 'prior_clicks', 'prior_ctr',
                   'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
X = X[model_features]
y = dataframe['future_decline_label'].values
groups = dataframe['client_hash_id']

# Same honest, client-grouped split as Week 6 (random_state=42, test_size=0.3)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
X_test_scaled = scaler.transform(X.iloc[test_idx])

model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
model.fit(X_train_scaled, y[train_idx])

# Scored only the test set (honest, unseen data) for ranking and reason code generation
test_dataframe = dataframe.iloc[test_idx].copy()
test_dataframe['model_probability'] = model.predict_proba(X_test_scaled)[:, 1]

# Reason codes
test_dataframe['prior_ctr'] = X.iloc[test_idx]['prior_ctr']
test_dataframe['high_visibility_at_risk'] = (
    (test_dataframe['prior_impressions'] >= 1000) & (test_dataframe['prior_avg_position'] > 10)
).astype(int)
test_dataframe['weak_position_signal'] = (
    test_dataframe['prior_avg_position'] > 10
).fillna(False).astype(int)
test_dataframe['low_prior_engagement'] = (
    (test_dataframe['prior_sessions'] > 0) & (test_dataframe['prior_engagement_rate'] < 0.30)
).fillna(False).astype(int)
test_dataframe['low_click_through_rate'] = (
    test_dataframe['prior_ctr'] < 0.01
).fillna(False).astype(int)
test_dataframe['limited_prior_visibility'] = (
    test_dataframe['prior_impressions'] < 1000
).astype(int)

def make_reason_code(row):
    reasons = []
    if row['high_visibility_at_risk']:
        reasons.append('high_visibility_at_risk')
    if row['weak_position_signal']:
        reasons.append('weak_position_signal')
    if row['low_prior_engagement']:
        reasons.append('low_prior_engagement')
    if row['low_click_through_rate']:
        reasons.append('low_click_through_rate')
    if row['limited_prior_visibility']:
        reasons.append('limited_prior_visibility')
    return '; '.join(reasons) if reasons else 'no_flag_triggered'

test_dataframe['reason_code'] = test_dataframe.apply(make_reason_code, axis=1)

# Rank by the model's predicted probability (honest, unseen-data score)
ranked = test_dataframe.sort_values(
    ['model_probability', 'prior_impressions'],
    ascending=[False, True],
).reset_index(drop=True)
ranked['rank'] = np.arange(1, len(ranked) + 1)

output_path = repo_root / 'work' / 'outputs' / 'march_model_ranked_queue.csv'
ranked_columns = [
    'rank',
    'client_hash_id',
    'content_hash_id',
    'model_probability',
    'reason_code',
    'prior_impressions',
    'prior_clicks',
    'prior_ctr',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
    'future_impressions',
    'future_decline_label',
]
ranked[ranked_columns].to_csv(output_path, index=False)

top_20 = ranked.head(20)
precision_at_20 = top_20['future_decline_label'].mean()
base_rate = ranked['future_decline_label'].mean()

metrics = pd.DataFrame({
    'metric': ['test_set_rows', 'future_decline_base_rate', 'precision_at_20'],
    'value': [len(ranked), base_rate, precision_at_20],
})

print(f"Saved model-ranked queue to: {output_path}")
print(metrics)
ranked[ranked_columns].head(10)

Saved model-ranked queue to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\march_model_ranked_queue.csv
                     metric        value
0             test_set_rows  28904.00000
1  future_decline_base_rate      0.18769
2           precision_at_20      0.35000


,rank,client_hash_id,content_hash_id,model_probability,reason_code,prior_impressions,prior_clicks,prior_ctr,prior_avg_position,prior_sessions,prior_engagement_rate,future_impressions,future_decline_label
0,1,client_62f4a7e64f5e0096,content_90abf1c28b62a2e4,0.429304,weak_position_signal; low_click_through_rate; ...,113.0,0.0,0.0,104.716814,0.0,NaN,53.0,1
1,2,client_62f4a7e64f5e0096,content_1e9fa51c07fee766,0.405470,weak_position_signal; low_click_through_rate; ...,103.0,0.0,0.0,92.106796,0.0,NaN,139.0,0
2,3,client_62f4a7e64f5e0096,content_7e4133c46bb7c589,0.399165,weak_position_signal; low_click_through_rate; ...,124.0,0.0,0.0,88.798387,0.0,NaN,49.0,1
3,4,client_62f4a7e64f5e0096,content_3a398db04bf7c610,0.391199,weak_position_signal; low_click_through_rate; ...,128.0,0.0,0.0,84.531250,0.0,NaN,13.0,1
4,5,client_62f4a7e64f5e0096,content_7ab0876ca393025c,0.379690,weak_position_signal; low_click_through_rate; ...,133.0,0.0,0.0,78.308271,0.0,NaN,1088.0,0
5,6,client_62f4a7e64f5e0096,content_edb3d41e07485a15,0.378435,weak_position_signal; low_click_through_rate; ...,132.0,0.0,0.0,77.621212,0.0,NaN,201.0,0
6,7,client_62f4a7e64f5e0096,content_c2870dfa1296ea52,0.375297,weak_position_signal; low_click_through_rate; ...,335.0,0.0,0.0,76.456716,0.0,NaN,1644.0,0
7,8,client_62f4a7e64f5e0096,content_3b88098d80da16b5,0.374190,weak_position_signal; low_click_through_rate; ...,317.0,0.0,0.0,75.801262,0.0,NaN,822.0,0
8,9,client_62f4a7e64f5e0096,content_af174291be2b1a26,0.373435,weak_position_signal; low_click_through_rate; ...,229.0,0.0,0.0,75.148472,0.0,NaN,262.0,0
9,10,client_62f4a7e64f5e0096,content_1424a94ca74e2388,0.372419,weak_position_signal; low_click_through_rate; ...,148.0,0.0,0.0,74.371622,0.0,NaN,288.0,0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:**  Account managers, customer success leads, and SEO specialists who manage large portfolios of web pages or content assets.

**For what:** Identifying pages showing early signs of traffic or engagement decline so proactive optimization or content refreshes can be deployed before rankings drop significantly.

**Where it stops being valid:** Highly seasonal pages (e.g., holiday-specific content), newly published pages with insufficient historical data, or pages experiencing sudden algorithmic shifts outside the model's training window.

In [2]:
import numpy as np

# How much of the eligible population this queue actually covers
population_size = len(dataframe)
test_size = len(ranked)
coverage_pct = test_size / population_size * 100

print(f"Eligible pages: {population_size}")
print(f"Pages in this honest queue (test split only): {test_size} ({coverage_pct:.1f}%)")

# Feature ranges seen in training vs. what the model is scoring here
train_ranges = X.iloc[train_idx][model_features].describe().loc[['min', 'max']]
test_ranges = X.iloc[test_idx][model_features].describe().loc[['min', 'max']]

print("\nTraining feature ranges:")
print(train_ranges)
print("\nTest (queue) feature ranges:")
print(test_ranges)

# Flag any test rows whose feature values fall outside the training range --
# the model is extrapolating for these, and predictions are less trustworthy there.
extrapolation_flags = pd.Series(False, index=ranked.index)
for feature in model_features:
    train_min = train_ranges.loc['min', feature]
    train_max = train_ranges.loc['max', feature]
    out_of_range = (ranked[feature] < train_min) | (ranked[feature] > train_max)
    extrapolation_flags = extrapolation_flags | out_of_range

ranked['extrapolation_risk'] = extrapolation_flags.astype(int)
print(f"\nRows scored outside the training feature range: "
      f"{ranked['extrapolation_risk'].sum()} of {len(ranked)}")

Eligible pages: 80322
Pages in this honest queue (test split only): 28904 (36.0%)

Training feature ranges:
     prior_impressions  prior_clicks  prior_ctr  prior_avg_position  \
min              100.0           0.0   0.000000            0.009865   
max           203401.0        3310.0   0.185185           85.468468   

     prior_sessions  prior_engagement_rate  
min             0.0                    0.0  
max          4038.0                    1.0  

Test (queue) feature ranges:
     prior_impressions  prior_clicks  prior_ctr  prior_avg_position  \
min              100.0           0.0   0.000000            0.000000   
max           160699.0         883.0   0.083333          104.716814   

     prior_sessions  prior_engagement_rate  
min             0.0                    0.0  
max            84.0                    1.0  

Rows scored outside the training feature range: 5 of 28904


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before anyone acts on this queue, three groups of rows need a second look — built in the cell below into a single `review_queue`:

1. **Unexplained high risk** — pages with `model_probability` ≥ 0.5 but no reason code fired (`reason_code == 'no_flag_triggered'`). The model sees something in the raw numbers that the plain-language flags don't explain; check the underlying metrics directly before acting.
2. **Low-confidence predictions** — pages with `model_probability` between 0.40 and 0.60. The model is essentially guessing here; treat these as "unknown," not "at risk."
3. **Extrapolated predictions** — pages flagged in Section 2 as having feature values outside the range the model was trained on. The model has no real basis for these scores.

**No-go list — never automate these, regardless of score:**
- Never auto-message, auto-pause, or auto-reprioritize a client's content based solely on `model_probability`.
- Never treat a single month's ranking as a performance judgment on a specific writer, editor, or client relationship.
- Never merge this queue with client billing, renewal, or contract decisions.
- Never present `model_probability` to a client as a guarantee or forecast — it's an internal triage aid.
- Never skip the reason-code sanity check for a page just because it ranks high — a top rank with a flimsy reason code is still worth a human look.

In [3]:
# Rows the model rates as high-risk but where none of the human-readable
# reason codes fired -- the "why" is opaque, so a person should look before acting.
unexplained_high_risk = ranked[
    (ranked['model_probability'] >= 0.5) & (ranked['reason_code'] == 'no_flag_triggered')
]

# Rows near the decision boundary (probability close to 0.5) -- the model is
# genuinely unsure here, not confidently right or wrong.
low_confidence = ranked[
    (ranked['model_probability'] >= 0.40) & (ranked['model_probability'] <= 0.60)
]

# Rows flagged in Section 2 as outside the training feature range.
extrapolated = ranked[ranked['extrapolation_risk'] == 1]

review_queue = (
    pd.concat([unexplained_high_risk, low_confidence, extrapolated])
    .drop_duplicates(subset=['client_hash_id', 'content_hash_id'])
    .sort_values('model_probability', ascending=False)
)

print(f"Unexplained high risk (no reason code fired): {len(unexplained_high_risk)}")
print(f"Low confidence (probability 0.40-0.60): {len(low_confidence)}")
print(f"Extrapolation risk (outside training feature range): {len(extrapolated)}")
print(f"Total rows requiring human review before any action: {len(review_queue)}")

no_go_columns = ['rank', 'client_hash_id', 'content_hash_id', 'model_probability', 'reason_code']
review_queue[no_go_columns].head(10)

Unexplained high risk (no reason code fired): 0
Low confidence (probability 0.40-0.60): 2
Extrapolation risk (outside training feature range): 5
Total rows requiring human review before any action: 5


,rank,client_hash_id,content_hash_id,model_probability,reason_code
0,1,client_62f4a7e64f5e0096,content_90abf1c28b62a2e4,0.429304,weak_position_signal; low_click_through_rate; ...
1,2,client_62f4a7e64f5e0096,content_1e9fa51c07fee766,0.405470,weak_position_signal; low_click_through_rate; ...
2,3,client_62f4a7e64f5e0096,content_7e4133c46bb7c589,0.399165,weak_position_signal; low_click_through_rate; ...
134,135,client_62f4a7e64f5e0096,content_f9257f44ed3a9b34,0.323596,weak_position_signal; low_click_through_rate; ...
12555,12556,client_62f4a7e64f5e0096,content_b3f11bfce6523f8f,0.247696,low_click_through_rate; limited_prior_visibility


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

I'm saving this run's metrics as a monitoring baseline (`work/outputs/model_monitoring_baseline.json`) so a future run has something concrete to compare against.

**Retrain triggers — any one of these should prompt a retrain or a fresh audit:**
- **Precision@20 drop:** more than 10 percentage points below this baseline's ~35% on a fresh month's honest test split.
- **Base rate drift:** the actual future-decline rate shifts more than 5 points from this baseline's ~18.8% — a sign the underlying environment (algorithm change, seasonal shift) moved, not just noise.
- **Population Stability Index (PSI) > 0.25** between the training feature distribution and a new month's feature distribution — implemented below and demonstrated on this run's own train/test split as a stand-in, since no second month is available yet to monitor against for real.
- **Coverage collapse:** if a future month's eligible page count or test-split size drops sharply, the queue may no longer be representative.

None of these are self-triggering — they're checks a person runs each cycle, not an automated alert system.

In [4]:
import json

# Snapshot of this run's metrics -- future runs compare against this baseline
# to catch drift before it erodes trust in the queue.
baseline_snapshot = {
    'run_date': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'base_rate': float(base_rate),
    'precision_at_20': float(precision_at_20),
    'test_set_rows': int(len(ranked)),
    'random_state': 42,
}

baseline_path = repo_root / 'work' / 'outputs' / 'model_monitoring_baseline.json'
with open(baseline_path, 'w') as f:
    json.dump(baseline_snapshot, f, indent=2)

print(f"Saved monitoring baseline to: {baseline_path}")
print(baseline_snapshot)


def population_stability_index(expected, actual, buckets=10):
    """
    Compares the distribution of a feature (or score) between two periods.
    A PSI above ~0.25 usually signals the population has shifted enough
    that the model's assumptions no longer hold -- a retrain trigger.
    """
    breakpoints = np.linspace(0, 100, buckets + 1)
    bucket_edges = np.percentile(expected, breakpoints)
    bucket_edges[0], bucket_edges[-1] = -np.inf, np.inf

    expected_pct = np.histogram(expected, bins=bucket_edges)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=bucket_edges)[0] / len(actual)

    expected_pct = np.where(expected_pct == 0, 1e-4, expected_pct)
    actual_pct = np.where(actual_pct == 0, 1e-4, actual_pct)

    return float(np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)))


# Demonstrated on this run's own train vs. test probability distributions,
# as a stand-in until a genuinely new month of data is available to monitor against.
train_probability = model.predict_proba(X_train_scaled)[:, 1]
demo_psi = population_stability_index(train_probability, ranked['model_probability'].values)
print(f"\nDemo PSI (train vs. test probability distribution): {demo_psi:.4f}")
print("Retrain trigger thresholds: PSI > 0.25, precision@20 drop > 10 points below baseline, "
      "or base rate drift > 5 points from this baseline's ~18.8%.")

Saved monitoring baseline to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\model_monitoring_baseline.json
{'run_date': '2026-08-21', 'base_rate': 0.1876902850816496, 'precision_at_20': 0.35, 'test_set_rows': 28904, 'random_state': 42}

Demo PSI (train vs. test probability distribution): 0.1219
Retrain trigger thresholds: PSI > 0.25, precision@20 drop > 10 points below baseline, or base rate drift > 5 points from this baseline's ~18.8%.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Everything below writes to `work/outputs/` and `work/figures/` so the capstone paper can reference these files directly instead of re-deriving them:
- `march_model_ranked_queue.csv` — the full ranked queue (already written in Section 1).
- `w07_summary_metrics.csv` — the base rate, precision@20, and test-set row count in one small table.
- `w07_probability_distribution.png` — a histogram of `model_probability` across the honest queue.
- `w07_top20_reason_codes.png` — how often each reason code appears among the top 20 ranked pages.

No raw per-client data leaves this notebook beyond what's already in the ranked queue CSV, which contains no client names — only pseudonymous `client_hash_id` values.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

figures_dir = repo_root / 'work' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

# Export the summary metrics as a small, paper-ready table (no raw data, just numbers)
metrics_export_path = repo_root / 'work' / 'outputs' / 'w07_summary_metrics.csv'
metrics.to_csv(metrics_export_path, index=False)
print(f"Saved summary metrics to: {metrics_export_path}")

# Figure 1: probability distribution across the honest queue
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ranked['model_probability'], bins=20, color='#4C72B0', edgecolor='white')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Model probability (predicted decline risk)')
ax.set_ylabel('Number of pages')
ax.set_title('Distribution of decline risk across the honest test queue')
fig.tight_layout()
fig_path_1 = figures_dir / 'w07_probability_distribution.png'
fig.savefig(fig_path_1, dpi=150)
plt.close(fig)
print(f"Saved figure: {fig_path_1}")

# Figure 2: how often each reason code appears among the top 20 ranked pages
reason_counts = (
    top_20['reason_code']
    .str.split('; ')
    .explode()
    .value_counts()
)

fig, ax = plt.subplots(figsize=(6, 4))
reason_counts.sort_values().plot(kind='barh', ax=ax, color='#DD8452')
ax.set_xlabel('Count among top 20 ranked pages')
ax.set_title('Reason codes behind the top 20 flagged pages')
fig.tight_layout()
fig_path_2 = figures_dir / 'w07_top20_reason_codes.png'
fig.savefig(fig_path_2, dpi=150)
plt.close(fig)
print(f"Saved figure: {fig_path_2}")

print("\nAll Section 5 exports:")
for p in [output_path, metrics_export_path, fig_path_1, fig_path_2]:
    print(f" - {p.relative_to(repo_root)}")

Matplotlib is building the font cache; this may take a moment.


Saved summary metrics to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\w07_summary_metrics.csv
Saved figure: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\figures\w07_probability_distribution.png
Saved figure: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\figures\w07_top20_reason_codes.png

All Section 5 exports:
 - work\outputs\march_model_ranked_queue.csv
 - work\outputs\w07_summary_metrics.csv
 - work\figures\w07_probability_distribution.png
 - work\figures\w07_top20_reason_codes.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.